# US Macro Regime Radar Results

This notebook centralizes the saved outputs of the project without rerunning the pipeline.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT / 'us_macro_regime_radar'
print('Project root:', ROOT)

def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

metrics_dir = ROOT / 'outputs' / 'metrics'
summaries_dir = ROOT / 'outputs' / 'summaries'
dashboard_dir = ROOT / 'outputs' / 'dashboard'

PHASE_COLORS = {
    'Expansion': '#1b9e77',
    'Slowdown': '#d95f02',
    'Contraction': '#d73027',
    'Recovery': '#4575b4',
}

def phase_segments(history: pd.DataFrame):
    segments = []
    if history.empty:
        return segments
    start_idx = 0
    for idx in range(1, history.shape[0]):
        if history.iloc[idx]['current_phase'] != history.iloc[idx - 1]['current_phase']:
            segments.append((history.iloc[start_idx]['date'], history.iloc[idx - 1]['date'], history.iloc[start_idx]['current_phase']))
            start_idx = idx
    segments.append((history.iloc[start_idx]['date'], history.iloc[-1]['date'], history.iloc[start_idx]['current_phase']))
    return segments

def add_phase_background(fig: go.Figure, history: pd.DataFrame, opacity: float = 0.20):
    for start_date, end_date, phase_name in phase_segments(history):
        fig.add_vrect(
            x0=start_date,
            x1=end_date + pd.offsets.MonthEnd(0),
            fillcolor=PHASE_COLORS.get(phase_name, '#cccccc'),
            opacity=opacity,
            line_width=0,
            layer='below',
        )


In [ ]:
fetch_summary = load_json(summaries_dir / 'fetch_macro_data_summary.json')
panel_summary = load_json(summaries_dir / 'build_monthly_panel_summary.json')
label_summary = load_json(summaries_dir / 'build_cycle_labels_summary.json')
phase_metrics = load_json(metrics_dir / 'phase_forecast_metrics.json')
risk_metrics = load_json(metrics_dir / 'recession_risk_metrics.json')
latest_snapshot = load_json(dashboard_dir / 'latest_snapshot.json')

summary_rows = pd.DataFrame([
    {'section': 'Fetch', 'value': fetch_summary['series_count']},
    {'section': 'Monthly panel rows', 'value': panel_summary['rows']},
    {'section': 'Feature count', 'value': panel_summary['feature_count']},
    {'section': 'Positive rate', 'value': round(label_summary['recession_within_6m_positive_rate'], 4)},
])
summary_rows


In [ ]:
phase_selection = pd.DataFrame(phase_metrics['validation_selection'])
risk_selection = pd.DataFrame(risk_metrics['validation_selection'])

display(phase_selection)
display(risk_selection)
print('Latest snapshot:', latest_snapshot)


In [ ]:
history = pd.DataFrame(load_json(dashboard_dir / 'history_payload.json')['rows'])
history['date'] = pd.to_datetime(history['date'])
plot_history = history[history['recession_risk_probability'].notna()].copy()
if plot_history.empty:
    plot_history = history.copy()

fig = go.Figure()
add_phase_background(fig, plot_history, opacity=0.22)

recession_months = plot_history[plot_history['is_recession_month'] == 1]
for _, row in recession_months.iterrows():
    fig.add_vrect(
        x0=row['date'],
        x1=row['date'] + pd.offsets.MonthEnd(0),
        fillcolor='#7f0000',
        opacity=0.10,
        line_width=0,
        layer='below',
    )

fig.add_trace(go.Scatter(
    x=plot_history['date'],
    y=plot_history['recession_risk_probability'],
    mode='lines',
    name='Recession within 6M probability',
    line={'color': '#111111', 'width': 2},
))
fig.update_layout(
    title='Historical recession-within-6m probability with phase background',
    xaxis_title='Date',
    yaxis_title='recession_risk_probability',
    template='plotly_white',
    hovermode='x unified',
)
fig.show()


In [ ]:
phase_order = ['Contraction', 'Recovery', 'Slowdown', 'Expansion']
phase_y = {'Contraction': 0, 'Recovery': 1, 'Slowdown': 2, 'Expansion': 3}
phase_strip = plot_history[['date', 'current_phase']].copy()
phase_strip['phase_y'] = phase_strip['current_phase'].map(phase_y)

fig = go.Figure()
for phase_name in phase_order:
    frame = phase_strip[phase_strip['current_phase'] == phase_name]
    if frame.empty:
        continue
    fig.add_trace(go.Scatter(
        x=frame['date'],
        y=frame['phase_y'],
        mode='markers',
        name=phase_name,
        marker={'symbol': 'square', 'size': 10, 'color': PHASE_COLORS[phase_name]},
        hovertemplate='%{x|%Y-%m}<br>' + phase_name + '<extra></extra>'
    ))
fig.update_layout(
    title='Historical phase strip',
    xaxis_title='Date',
    yaxis_title='Phase',
    yaxis={'tickmode': 'array', 'tickvals': [0, 1, 2, 3], 'ticktext': phase_order},
    template='plotly_white',
    height=280,
)
fig.show()


In [ ]:
forecast = pd.DataFrame(load_json(dashboard_dir / 'forecast_payload.json')['phase_probabilities_long'])
fig = px.line(
    forecast,
    x='horizon_months',
    y='probability',
    color='phase',
    markers=True,
    color_discrete_map=PHASE_COLORS,
    title='Forward phase probability path',
    template='plotly_white'
)
fig.show()
